In [1]:
#paths
csv_path = "/home/theo/PycharmProjects/Bookoflife/Original/data/raw/data.csv"
cdb_path = "/home/theo/PycharmProjects/Bookoflife/Original/data/raw/data.cdb"
processed = "home/theo/PycharmProjects/Bookoflife/Original/data/processed"

In [2]:
import re
import pandas as pd
from pathlib import Path

#parser from marcus
def parse_cdb(cdb_path):
    """Parse NLSY79 codebook (.cdb) - simple version"""
    codebook = {}
    text = Path(cdb_path).read_text()

    # Split by long dashes
    blocks = re.split(r"-{40,}", text)

    for block in blocks:
        # Match header: H00016.00    [H40-BPAR-1]    Survey Year: XRND
        m = re.match(r"\s*(\w+\.\d+)\s+\[([^\]]+)\]\s+Survey Year:\s*(\S+)", block)
        if not m:
            continue

        rnum = m.group(1).replace(".", "")  # H0001600
        qname = m.group(2)                   # H40-BPAR-1
        year = m.group(3)                    # XRND

        # Extract question (after "PRIMARY VARIABLE")
        question = ""
        qm = re.search(r"PRIMARY VARIABLE\s*\n\s*\n\s*(.+?)(?:\n\s*\n|$)", block, re.DOTALL)
        if qm:
            question = qm.group(1).strip().replace("\n", " ")

        codebook[rnum] = {
            "rnum": rnum,
            "qname": qname,
            "year": year,
            "question": question,
        }

    return codebook


In [3]:
#parse through codebook
codebook = parse_cdb(cdb_path)

cb_df = pd.DataFrame(codebook.values())


In [4]:
#identify variables with year withput int
invalid_year = ~cb_df["year"].astype(str).str.fullmatch(r"\d{4}")
invalid_year_df = cb_df[invalid_year]
#identify variables with unique qname (asked only once across all years)
unique_qname = cb_df.groupby("qname")["qname"].transform("size") == 1

# retroperspecitv questions
#filter for year 1988, and for questions which start with lived with, and lived in
living_struc = (
    cb_df["year"].astype(str).eq("1988")
    & cb_df["question"]
        .astype(str)
        .str.lower()
        .str.startswith(("lived with", "lived in"), na=False)
)

##sind die ganzen 1988 struktur variablen auch drinnen -> wie identifizieren wir die? questinon text ist mit lived with..
cbd_p = cb_df[unique_qname & ~living_struc | invalid_year] #XRND variablen raus da kein informationsverlust
cbd_pl = cb_df[~(invalid_year | unique_qname) | (cb_df["rnum"] == "R0000100")]
cbd_ls = cb_df[living_struc | (cb_df["rnum"] == "R0000100")]

In [5]:
invalid_year_df

,rnum,qname,year,question
0,H0001600,H40-BPAR-1,XRND,R'S BIOLOGICAL FATHER LIVING?
1,H0002400,H40-BPAR-6,XRND,R'S BIOLOGICAL MOTHER LIVING?
2,H0013600,H50BPAR-1,XRND,R'S BIOLOGICAL FATHER LIVING?
3,H0014700,H50BPAR-6,XRND,R'S BIOLOGICAL MOTHER LIVING?
4,H0046300,H60BPAR-1,XRND,R'S BIOLOGICAL FATHER LIVING?
5,H0048900,H60BPAR-6,XRND,R'S BIOLOGICAL MOTHER LIVING?
17,R0214700,SAMPLE_RACE,78SCRN,R'S RACIAL/ETHNIC COHORT FROM SCREENER


In [6]:
print(len(cbd_p))
print("+", len(cbd_pl))
print("+", len(cbd_ls))
print("=", len(cbd_ls) + len(cbd_pl) +len (cbd_p), '=', len(cb_df), "-", len(invalid_year_df), "+", 1, "+", 2, "=", len(cb_df) - len(invalid_year_df) + 1 + 2)
#-7 + for all XRND variables, +2 because case_id is in all three cbd_*


31
+ 181
+ 255
= 467 = 465 - 7 + 1 + 2 = 461


In [7]:
#lode data with korrekt rnmus
df_csv = pd.read_csv(csv_path)
csv_pl = df_csv[cbd_pl["rnum"]]
csv_p = df_csv[cbd_p["rnum"]]
csv_ls = df_csv[cbd_ls["rnum"]]

In [8]:
rnum_to_qname = dict(zip(cb_df['rnum'], cb_df['qname']))
p = csv_p.rename(columns=rnum_to_qname)
p = p.rename(columns={"CASEID": "caseid"})
p["birthyear"] =  1979 - p["FAM-1B"]
p = p.drop(columns='FAM-1B')


In [9]:
rnum_vars = cbd_pl["rnum"].tolist()
print(len(rnum_vars))
melted = csv_pl.melt(id_vars=["R0000100"], value_vars=rnum_vars,
                     var_name="rnum", value_name="value")

melted = melted.merge(cb_df[["rnum", "qname", "year", "question"]],
                      on="rnum", how="left")


melted = melted.rename(columns={"R0000100": "caseid"})
melted = melted.sort_values(["caseid", "rnum"]).reset_index(drop=True)


melted["syear"] = pd.to_numeric(melted["year"], errors="coerce")


181


In [10]:
pivoted_pl = melted.pivot_table(
    index=['caseid', 'year'],
    columns='qname',
    values='value',
).reset_index()


In [11]:
rnum_vars_ls = cbd_ls["rnum"].tolist()

# melt: von wide zu long. id_vars ist  'R0000100'
melted_ls = csv_ls.melt(id_vars=["R0000100"], value_vars=rnum_vars_ls,
                     var_name="rnum", value_name="value")

melted_ls = melted_ls.merge(cb_df[["rnum", "qname", "year", "question"]],
                      on="rnum", how="left")

melted_ls = melted_ls.rename(columns={"R0000100": "caseid"})
melted_ls = melted_ls.sort_values(["caseid", "rnum"]).reset_index(drop=True)

melted_ls["syear"] = pd.to_numeric(melted_ls["year"], errors="coerce")

In [12]:
pivoted_ls = melted_ls.pivot_table(
    index=['caseid', 'year'],
    columns='qname',
    values='value',
).reset_index()

In [13]:

def reshape_retrospective(df, var_prefix):
    df = df.dropna(subset=['birthyear'])

    cols = []
    for col in df.columns:
        if col.startswith(var_prefix.upper()):
            # Extrahiere die Zahl am Ende
            match = re.search(r'(\d+)$', col)
            if match:
                age_num = int(match.group(1))
                if 0 <= age_num <= 18:
                    cols.append(col)
    if not cols:
        print(f"Keine passenden Spalten gefunden für Präfix '{var_prefix}'")
        return pd.DataFrame()

    long = df.melt(
        id_vars=['caseid', 'year', 'birthyear'],
        value_vars=cols,
        var_name='suffix',
        value_name=var_prefix
    )

    long['age'] = long['suffix'].str.extract(r'(\d+)$').astype(int)
    long['year'] = long['birthyear'] + long['age']  # ✓ direkt als 'year'

    return long[['caseid', 'year', 'age', 'birthyear', var_prefix]].sort_values(['caseid', 'year']).reset_index(drop=True)

In [14]:
# Merge with caseid  year
pivoted_ls_with_birth = pivoted_ls.merge(
    p[['caseid', 'birthyear']],  # Nur eindeutige caseid-birthyear
    on='caseid',
    how='left'
)


In [15]:
# Select all variables ending in 18, as we only need those that are available for all age groups from 0 to 18 in order to convert them to long format
cols_ending_18 = [col for col in pivoted_ls_with_birth.columns
                  if re.search(r'18$', col)]
#isolate prefixes
prefixes = [re.sub(r'\d+$', '', col) for col in cols_ending_18]

#loop reshape_retro.. over all isolated prefixes
long_dfs = [reshape_retrospective(pivoted_ls_with_birth, prefix) for prefix in prefixes]

result_ls = pd.concat(long_dfs, axis=1)
result_ls = result_ls.loc[:, ~result_ls.columns.duplicated()].sort_values(['caseid', 'year']).reset_index(drop=True)


In [16]:
#convert year for all datasets to string
result_ls['year'] = result_ls['year'].astype(int)
pivoted_pl['year'] = pivoted_pl['year'].astype(int)

#merge ls with pivoted_pl
merged_1 = result_ls.merge(pivoted_pl, on=['caseid', 'year'], how='outer')
merged_1 = merged_1.drop(columns=['age', 'birthyear'])


In [17]:

final_result = merged_1.merge(
    p,
    on=['caseid'],
    how='left',
)
final_result["age"] = final_result["year"] - final_result["birthyear"]


In [18]:
final_result

,caseid,year,ADOPTD,ADOPTM,BDAD,BMOM,CHLDHM,DTENT,FOSTER,FRIEND,...,Q9-88.06,Q9-89.07,Q9-89.08,Q9-90.04,Q9-91.07,Q9-91.08,Q9-89.10,Q9-91.10,birthyear,age
0,1,1959,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,...,-5,-5,-5,-5,-5,-5,-5,-5,1959,0
1,1,1960,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,...,-5,-5,-5,-5,-5,-5,-5,-5,1959,1
2,1,1961,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,...,-5,-5,-5,-5,-5,-5,-5,-5,1959,2
3,1,1962,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,...,-5,-5,-5,-5,-5,-5,-5,-5,1959,3
4,1,1963,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,-5.0,...,-5,-5,-5,-5,-5,-5,-5,-5,1959,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
507323,12686,2010,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-5,-5,-5,-5,-5,-5,-5,-5,1961,49
507324,12686,2012,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-5,-5,-5,-5,-5,-5,-5,-5,1961,51
507325,12686,2014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-5,-5,-5,-5,-5,-5,-5,-5,1961,53
507326,12686,2016,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-5,-5,-5,-5,-5,-5,-5,-5,1961,55


In [19]:

# Erstelle eine "schlanke" Version: Vergleiche mit der Zeile darüber (nicht Baseline)
def to_sparse_format(df):
    """Nur Änderungen zwischen aufeinanderfolgenden Zeilen"""
    result = []

    for caseid in df['caseid'].unique():
        person_df = df[df['caseid'] == caseid].sort_values('year').reset_index(drop=True)

        for i, row in person_df.iterrows():
            sparse_row = {'caseid': caseid, 'year': row['year']}

            # Erste Zeile: immer alle Werte nehmen
            if i == 0:
                for col in person_df.columns:
                    if col not in ['caseid', 'year']:
                        sparse_row[col] = row[col]
            else:
                # Vergleiche mit vorheriger Zeile
                prev_row = person_df.iloc[i-1]
                for col in person_df.columns:
                    if col not in ['caseid', 'year']:
                        # Nimm Wert wenn: unterschiedlich von vorher ODER neu vorhanden (war NaN, ist jetzt nicht)
                        if pd.isna(prev_row[col]) or pd.isna(row[col]) or prev_row[col] != row[col]:
                            sparse_row[col] = row[col]

            result.append(sparse_row)

    return pd.DataFrame(result)

merged_1_sparse = to_sparse_format(merged_1)
merged_1_sparse

KeyboardInterrupt: 

In [ ]:
%cd processed

final_result.to_csv("nlsy79_reshaped_pl_p.csv", index=False)
merged_1.to_csv("nlsy79_pl.csv", index=False)
p.to_csv("nlsy79_p.csv", index=False)
merged_1_sparse.to_csv("nlsy79_pl_sparse.csv", index=False)